# Play Call Predictor — Model 1 (MDP Architecture)

**What this notebook does:**
1. Loads NFL play-by-play data from `nfl_data_py` (2018–2024, ~1.5M plays)
2. Engineers game-state features: down, distance bucket, yard line zone, score diff bucket, quarter, time remaining bucket, offense/defense formation tendency
3. Labels each play as one of 4 classes: **run**, **pass**, **punt**, **field_goal**
4. Trains a lightweight MLP classifier on T4 GPU
5. Validates that predicted probabilities match real NFL play-call rates
6. Saves weights + encoder artifacts to Google Drive (same pattern as TabSyn)

**Before running:** Switch runtime to T4 GPU — Runtime → Change runtime type → T4 GPU

**Output artifacts (download and place in `backend/python_backend/play_call_weights/`):**
- `play_call_model.pt` — PyTorch MLP weights
- `play_call_label_encoder.pkl` — LabelEncoder for play type classes
- `play_call_feature_meta.json` — feature names, bucket boundaries, category maps
- `play_call_config.json` — model architecture config for inference service

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Switch to T4 GPU runtime before proceeding.')

## 1: Install dependencies

In [ ]:
%%capture
!pip install nfl_data_py scikit-learn pandas numpy torch --quiet

## 2: Load play-by-play data

In [ ]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 2018-2024: ~1.5M rows, covers modern spread era
SEASONS = list(range(2018, 2025))
print(f'Loading PBP for seasons: {SEASONS}')
pbp_raw = nfl.load_pbp(SEASONS)
print(f'Loaded: {pbp_raw.shape[0]:,} plays x {pbp_raw.shape[1]} columns')

## 3: Filter to meaningful scrimmage plays

Keep only real offensive plays: run, pass, punt, field goal attempt.  
Drop kickoffs, PATs, spikes, kneels, penalties-only plays.

In [ ]:
KEEP_TYPES = {'run', 'pass', 'punt', 'field_goal'}

df = pbp_raw[
    pbp_raw['play_type'].isin(KEEP_TYPES) &
    pbp_raw['down'].notna() &
    pbp_raw['ydstogo'].notna() &
    pbp_raw['yardline_100'].notna() &
    pbp_raw['score_differential'].notna() &
    pbp_raw['qtr'].notna() &
    pbp_raw['game_seconds_remaining'].notna()
].copy()

print(f'After filtering: {len(df):,} plays')
print('\nPlay type distribution:')
dist = df['play_type'].value_counts(normalize=True) * 100
for pt, pct in dist.items():
    print(f'  {pt:<12} {pct:.1f}%')

## 4: Feature engineering

Convert raw game-state variables into bucketed integer features for the MLP.  
All features become integers so we can use embeddings for each dimension.

In [ ]:
# --- Distance bucket ---
# short (1-2), medium (3-6), long (7-10), very_long (11+)
def dist_bucket(x):
    if x <= 2:  return 0  # short
    elif x <= 6: return 1  # medium
    elif x <= 10: return 2  # long
    else:        return 3  # very_long

# --- Yard line zone ---
# own_deep (1-20), own_mid (21-40), midfield (41-60),
# opp_mid (61-80), red_zone (81-95), goal_line (96-100)
# yardline_100 = yards from OWN end zone (so 100 = opp goal line)
def yard_zone(x):
    # nfl_data_py: yardline_100 = yards to OPPONENT end zone
    if x >= 80:   return 0  # own deep
    elif x >= 60: return 1  # own mid
    elif x >= 40: return 2  # midfield
    elif x >= 20: return 3  # opp mid
    elif x >= 5:  return 4  # red zone
    else:         return 5  # goal line

# --- Score differential bucket ---
# large_deficit (<=-17), deficit (-16 to -7), close (-6 to 6),
# lead (7 to 16), large_lead (>=17)
def score_bucket(x):
    if x <= -17:  return 0
    elif x <= -7: return 1
    elif x <= 6:  return 2
    elif x <= 16: return 3
    else:         return 4

# --- Time remaining bucket (game_seconds_remaining) ---
# 0=clutch(<2min Q4/OT), 1=late(2-8min Q4), 2=mid(Q3+early Q4),
# 3=first_half, 4=early(Q1)
def time_bucket(row):
    secs = row['game_seconds_remaining']
    qtr = row['qtr']
    if secs <= 120 and qtr >= 4:  return 0  # clutch
    elif secs <= 480 and qtr == 4: return 1  # late Q4
    elif qtr == 3 or (qtr == 4 and secs > 480): return 2  # second half mid
    elif qtr == 2:                return 3  # first half
    else:                         return 4  # early Q1

print('Applying feature buckets...')
df['feat_down']       = df['down'].astype(int) - 1          # 0-3
df['feat_dist']       = df['ydstogo'].apply(dist_bucket)    # 0-3
df['feat_zone']       = df['yardline_100'].apply(yard_zone) # 0-5
df['feat_score']      = df['score_differential'].apply(score_bucket)  # 0-4
df['feat_qtr']        = (df['qtr'].clip(1, 5) - 1).astype(int)        # 0-4
df['feat_time']       = df.apply(time_bucket, axis=1)       # 0-4

# Half flag: important for punt/FG decision-making near end of half
df['feat_half_end']   = (
    ((df['qtr'] == 2) & (df['game_seconds_remaining'] <= 1860)) |
    ((df['qtr'] >= 4) & (df['game_seconds_remaining'] <= 120))
).astype(int)  # 0-1

# Whether the offense is trailing (affects aggression on 4th down)
df['feat_trailing']   = (df['score_differential'] < 0).astype(int)  # 0-1

FEATURE_COLS = [
    'feat_down', 'feat_dist', 'feat_zone', 'feat_score',
    'feat_qtr', 'feat_time', 'feat_half_end', 'feat_trailing'
]

# Cardinality of each feature (for embedding layer sizes)
FEAT_CARDINALITY = {
    'feat_down': 4,
    'feat_dist': 4,
    'feat_zone': 6,
    'feat_score': 5,
    'feat_qtr': 5,
    'feat_time': 5,
    'feat_half_end': 2,
    'feat_trailing': 2,
}

print('Feature engineering done.')
print(df[FEATURE_COLS].head(3))

## 5: Encode labels

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pickle, json, os

le = LabelEncoder()
df['label'] = le.fit_transform(df['play_type'])

print('Classes:', list(le.classes_))
print('Label mapping:', {k: int(v) for k, v in zip(le.classes_, le.transform(le.classes_))})

N_CLASSES = len(le.classes_)
print(f'\nN_CLASSES: {N_CLASSES}')

## 6: Train/val split

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLS].values.astype(np.int64)
y = df['label'].values.astype(np.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')
print('Train class distribution:')
for cls, lbl in zip(le.classes_, range(N_CLASSES)):
    pct = (y_train == lbl).sum() / len(y_train) * 100
    print(f'  {cls:<12} {pct:.1f}%')

## 7: Define the MLP model

Each categorical feature gets its own embedding layer (same pattern as TabSyn).  
Embeddings are concatenated and passed through 3 hidden layers → softmax over 4 play types.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

EMB_DIM = 8   # embedding dimension per feature
HIDDEN  = 256
DROPOUT = 0.2


class PlayCallMLP(nn.Module):
    def __init__(self, feat_cardinality: dict, emb_dim: int, hidden: int, n_classes: int):
        super().__init__()
        self.feat_names = list(feat_cardinality.keys())
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, emb_dim)
            for cardinality in feat_cardinality.values()
        ])
        in_dim = len(feat_cardinality) * emb_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.SiLU(),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, n_features) long tensor
        embs = [self.embeddings[i](x[:, i]) for i in range(len(self.feat_names))]
        h = torch.cat(embs, dim=-1)
        return self.net(h)  # logits

    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.forward(x), dim=-1)


model = PlayCallMLP(FEAT_CARDINALITY, EMB_DIM, HIDDEN, N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')
print(model)

## 8: Train

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS     = 40
BATCH_SIZE = 4096
LR         = 3e-3

X_tr_t = torch.LongTensor(X_train).to(DEVICE)
y_tr_t = torch.LongTensor(y_train).to(DEVICE)
X_vl_t = torch.LongTensor(X_val).to(DEVICE)
y_vl_t = torch.LongTensor(y_val).to(DEVICE)

train_ds = TensorDataset(X_tr_t, y_tr_t)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_dl), epochs=EPOCHS
)

best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_vl_t)
        val_loss = F.cross_entropy(val_logits, y_vl_t).item()
        val_acc = (val_logits.argmax(dim=-1) == y_vl_t).float().mean().item()

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | train_loss {total_loss/len(train_dl):.4f} '
              f'| val_loss {val_loss:.4f} | val_acc {val_acc*100:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/tmp/play_call_model_best.pt')

print(f'\nBest val accuracy: {best_val_acc*100:.2f}%')

## 9: Per-class accuracy + calibration check

The model's predicted probabilities for each game state should roughly match  
real NFL play-call rates. We check a few canonical situations.

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('/tmp/play_call_model_best.pt', map_location=DEVICE))
model.eval()

# Per-class accuracy on validation set
with torch.no_grad():
    preds = model(X_vl_t).argmax(dim=-1).cpu().numpy()

print('Per-class accuracy:')
for i, cls in enumerate(le.classes_):
    mask = y_val == i
    if mask.sum() == 0:
        continue
    acc = (preds[mask] == i).mean() * 100
    count = mask.sum()
    print(f'  {cls:<12} acc={acc:.1f}%  n={count:,}')

print()

# Calibration: canonical NFL situations
# Format: (down, dist_bucket, zone, score_bucket, qtr, time_bucket, half_end, trailing)
# Classes order from le.classes_ printed above
SCENARIOS = [
    # (label, feature_vector)
    ('1st & 10, own 25, tied, Q1',
     [0, 2, 1, 2, 0, 4, 0, 0]),   # expect: ~55% pass, ~45% run
    ('2nd & 3, own 40, tied, Q2',
     [1, 0, 1, 2, 1, 3, 0, 0]),   # expect: ~50% run
    ('3rd & 8, own 30, trailing, Q4',
     [2, 3, 1, 1, 3, 1, 0, 1]),   # expect: heavy pass ~85%
    ('4th & 2, opp 38, close, Q4',
     [3, 0, 3, 2, 3, 1, 0, 0]),   # expect: punt heavy but some go-for-it
    ('4th & 5, opp 18, tied, Q4',
     [3, 1, 4, 2, 3, 1, 0, 0]),   # expect: FG ~70%
    ('3rd & 10, own 10, large deficit, Q4',
     [2, 3, 0, 0, 3, 1, 0, 1]),   # expect: pass ~90%
    ('2nd & 1, opp 1, leading, Q3',
     [1, 0, 5, 3, 2, 2, 0, 0]),   # expect: run heavy (goal line)
    ('4th & 12, own 25, late, Q4 (clutch)',
     [3, 3, 0, 1, 3, 0, 1, 1]),   # trailing desperate: pass
]

print(f'Calibration check (classes: {list(le.classes_)}):')
print(f'{"Scenario":<42} ' + '  '.join(f"{c:<9}" for c in le.classes_))
print('-' * 90)

for label, feats in SCENARIOS:
    x = torch.LongTensor([feats]).to(DEVICE)
    with torch.no_grad():
        probs = model.predict_proba(x).cpu().numpy()[0]
    prob_str = '  '.join(f'{p*100:6.1f}%  ' for p in probs)
    print(f'{label:<42} {prob_str}')

## 10: Mount Drive and save all artifacts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/nfl_data/play_call_weights'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Saving to:', SAVE_DIR)

In [ ]:
import shutil

# 1. Model weights
shutil.copy('/tmp/play_call_model_best.pt', f'{SAVE_DIR}/play_call_model.pt')
print('Saved play_call_model.pt')

# 2. Label encoder
with open(f'{SAVE_DIR}/play_call_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print('Saved play_call_label_encoder.pkl')

# 3. Feature metadata (bucket boundaries + feature names)
feature_meta = {
    'feature_cols': FEATURE_COLS,
    'feat_cardinality': FEAT_CARDINALITY,
    'classes': list(le.classes_),
    'bucket_definitions': {
        'feat_down': {
            'description': 'NFL down (1-4) encoded as 0-3',
            'values': {0: '1st', 1: '2nd', 2: '3rd', 3: '4th'}
        },
        'feat_dist': {
            'description': 'Yards to go bucket',
            'values': {0: 'short(1-2)', 1: 'medium(3-6)', 2: 'long(7-10)', 3: 'very_long(11+)'}
        },
        'feat_zone': {
            'description': 'Field position zone (yardline_100 = yards to opp end zone)',
            'values': {0: 'own_deep(80-100)', 1: 'own_mid(60-79)', 2: 'midfield(40-59)',
                       3: 'opp_mid(20-39)', 4: 'red_zone(5-19)', 5: 'goal_line(1-4)'}
        },
        'feat_score': {
            'description': 'Score differential bucket (offense - defense)',
            'values': {0: 'large_deficit(<=-17)', 1: 'deficit(-16 to -7)',
                       2: 'close(-6 to 6)', 3: 'lead(7 to 16)', 4: 'large_lead(>=17)'}
        },
        'feat_qtr': {
            'description': 'Quarter encoded as 0-4 (OT=4)',
            'values': {0: 'Q1', 1: 'Q2', 2: 'Q3', 3: 'Q4', 4: 'OT'}
        },
        'feat_time': {
            'description': 'Time remaining urgency bucket',
            'values': {0: 'clutch(<2min Q4)', 1: 'late_Q4(2-8min)',
                       2: 'second_half_mid', 3: 'first_half', 4: 'early_Q1'}
        },
        'feat_half_end': {
            'description': 'Near end of half flag',
            'values': {0: 'normal', 1: 'near_half_end'}
        },
        'feat_trailing': {
            'description': 'Offense trailing flag',
            'values': {0: 'not_trailing', 1: 'trailing'}
        },
    }
}

with open(f'{SAVE_DIR}/play_call_feature_meta.json', 'w') as f:
    json.dump(feature_meta, f, indent=2)
print('Saved play_call_feature_meta.json')

# 4. Model config for inference microservice
model_config = {
    'emb_dim': EMB_DIM,
    'hidden': HIDDEN,
    'dropout': DROPOUT,
    'n_classes': N_CLASSES,
    'n_features': len(FEATURE_COLS),
    'best_val_acc': best_val_acc,
    'training_seasons': SEASONS,
}
with open(f'{SAVE_DIR}/play_call_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print('Saved play_call_config.json')

print(f'\nAll artifacts saved to: {SAVE_DIR}')
for fname in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{fname}')
    print(f'  {fname:<45} {size/1024:.1f} KB')

## 11: Download weights

Zip everything and download. Place the contents of the zip in `backend/python_backend/play_call_weights/`.

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/play_call_weights'
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
files.download(f'{zip_path}.zip')
print('Download started. Extract and place in backend/python_backend/play_call_weights/')

## Appendix: Quick inference test

Verify the model loads correctly and produces reasonable probabilities  
from scratch (simulates what the inference microservice will do).

In [ ]:
# Simulate cold load (what the inference microservice does at startup)
with open(f'{SAVE_DIR}/play_call_config.json') as f:
    cfg = json.load(f)
with open(f'{SAVE_DIR}/play_call_feature_meta.json') as f:
    meta = json.load(f)
with open(f'{SAVE_DIR}/play_call_label_encoder.pkl', 'rb') as f:
    le_loaded = pickle.load(f)

model_loaded = PlayCallMLP(
    feat_cardinality=meta['feat_cardinality'],
    emb_dim=cfg['emb_dim'],
    hidden=cfg['hidden'],
    n_classes=cfg['n_classes'],
).to(DEVICE)
model_loaded.load_state_dict(torch.load(f'{SAVE_DIR}/play_call_model.pt', map_location=DEVICE))
model_loaded.eval()

print('Cold-load successful. Running test inference...')
print()

# 3rd & 8 from own 25, tied, Q3 — expect heavy pass
test_feats = [2, 2, 1, 2, 2, 2, 0, 0]  # down=3rd, dist=long, zone=own_mid, score=close, qtr=Q3
x_test = torch.LongTensor([test_feats]).to(DEVICE)
with torch.no_grad():
    probs = model_loaded.predict_proba(x_test).cpu().numpy()[0]

print('3rd & 8, own 25, tied, Q3:')
for cls, prob in zip(le_loaded.classes_, probs):
    bar = '#' * int(prob * 40)
    print(f'  {cls:<12} {prob*100:5.1f}%  {bar}')

print('\nInference microservice is ready to be wired in.')